Cell 0 — Fix environment + install deps (run once, then RESTART kernel)

In [1]:
import sys, subprocess

def pip(cmd):
    print(">", "pip", cmd)
    subprocess.check_call([sys.executable, "-m", "pip"] + cmd.split())

pip("install -U -q pip setuptools wheel")

# Repair numpy/scipy/sklearn if your env is broken (you had _center import errors before)
def sci_ok():
    try:
        import numpy as np
        import scipy
        import sklearn
        return True
    except Exception as e:
        print("⚠️ scientific stack broken:", repr(e))
        return False

if not sci_ok():
    pip("uninstall -y -q numpy scipy scikit-learn")
    pip("install -q --no-cache-dir --force-reinstall numpy==1.26.4 scipy==1.12.0 scikit-learn==1.4.2 pandas==2.2.2")

# HF stack
pip("install -q -U transformers==4.45.2 datasets==2.21.0 accelerate==0.34.2 evaluate==0.4.2 sentencepiece")

# For 4-bit LLM (Kaggle GPU only)
try:
    import bitsandbytes  # noqa
except Exception:
    try:
        pip("install -q -U bitsandbytes")
    except Exception:
        print("⚠️ bitsandbytes not available (CPU-only). We'll fall back to a smaller explainer.")

print("✅ Done. Now RESTART the runtime/kernel, then run the next cell.")


> pip install -U -q pip setuptools wheel
> pip install -q -U transformers==4.45.2 datasets==2.21.0 accelerate==0.34.2 evaluate==0.4.2 sentencepiece
✅ Done. Now RESTART the runtime/kernel, then run the next cell.


Cell 1 — Config

In [ ]:
from pathlib import Path

# --- Paths ---
DATA_ROOT = "/kaggle/input"
LOCAL_TRAIN_FALLBACK = "/mnt/data/train.csv"   # if you run locally

# --- High-accuracy classifier ---
CLASSIFIER_MODEL = "microsoft/deberta-v3-large"

# --- Best explainer LLM (teacher feedback) ---
EXPLAINER_MODEL = "Qwen/Qwen2.5-7B-Instruct"        # open + strong
EXPLAINER_FALLBACK_MODEL = "google/flan-t5-large"   # if no 4-bit GPU

# ============================================================
# OPTIMIZED TRAINING CONFIG FOR HIGHEST ACCURACY
# ============================================================

# --- Training ---
N_FOLDS = 5
SEEDS = [42, 123, 456]     # Multi-seed ensemble (3x better stability)
EPOCHS = 4                 # More epochs = better convergence (try 5 if time allows)
LR = 1.5e-5                # Slightly lower LR for stability
BATCH_SIZE = 4             # DeBERTa-large is heavy. Use 2-4 on T4.
GRAD_ACCUM = 8             # Larger effective batch (4*8=32) improves stability
MAX_LEN = 512              # Longer context captures more info
WEIGHT_DECAY = 0.02        # Slightly stronger regularization
USE_FP16 = True

# --- NEW: Advanced training techniques ---
WARMUP_RATIO = 0.1         # 10% warmup for LR scheduler
LABEL_SMOOTHING = 0.1      # Reduces overconfidence, improves generalization
USE_LAYERWISE_LR_DECAY = True   # Lower LR for earlier layers (0.95 decay)
LAYERWISE_LR_DECAY = 0.95
MAX_GRAD_NORM = 1.0        # Gradient clipping

# --- NEW: Advanced regularization ---
USE_AWP = True             # Adversarial Weight Perturbation (big accuracy boost!)
AWP_START_EPOCH = 1        # Start AWP after epoch 1
AWP_LR = 0.1
AWP_EPS = 1e-4

USE_RDROP = True           # R-Drop consistency regularization
RDROP_ALPHA = 0.5          # KL divergence weight

USE_EMA = True             # Exponential Moving Average
EMA_DECAY = 0.999

# --- Accuracy boosts (safe vs risky) ---
USE_FAMILY_FILTER = True            # uses inferred correctness to filter True/False label families (usually helps)
USE_QID_LABEL_RESTRICTION = False   # can inflate CV; keep False unless you explicitly want it

# --- NEW: Post-processing ---
USE_TEMPERATURE_SCALING = True      # Calibrate probabilities
TEMPERATURE = 1.5

# --- Explainability ---
USE_LLM_EXPLAINER = True
USE_LLM_VERIFIER = True
KB_BUILD_TOP_N = 200
KB_EXAMPLES_PER_MISCON = 6

# --- Cache ---
WORKDIR = Path("./")
KB_PATH = WORKDIR / "misconception_kb.json"


Cell 2 — Load train.csv + detect columns

In [3]:
import pandas as pd

def find_file(filename: str, root: str) -> str:
    rootp = Path(root)
    hits = sorted(rootp.rglob(filename), key=lambda p: len(str(p)))
    return str(hits[0]) if hits else ""

train_path = find_file("train.csv", DATA_ROOT)
if not train_path and Path(LOCAL_TRAIN_FALLBACK).exists():
    train_path = LOCAL_TRAIN_FALLBACK
if not train_path:
    raise FileNotFoundError("train.csv not found. Put it in /kaggle/input/** or set LOCAL_TRAIN_FALLBACK.")

train_df = pd.read_csv(train_path)
print("train:", train_path, train_df.shape)
print("cols:", list(train_df.columns))

def lower_map(cols): return {c.lower(): c for c in cols}
cols = lower_map(train_df.columns)

cat_col = cols.get("category")
mis_col = cols.get("misconception")
if cat_col is None or mis_col is None:
    raise ValueError("Need columns: Category, Misconception")

qid_col = None
for k in ["question_id","questionid","item_id","itemid","problem_id","problemid","stem_id","stemid"]:
    if k in cols: qid_col = cols[k]; break

ans_col = None
for k in ["mc_answer","answer_choice","answer","choice","selected_answer","student_answer","option"]:
    if k in cols: ans_col = cols[k]; break

exp_col = None
for k in ["student_explanation","explanation","student_response","response","open_response","rationale"]:
    if k in cols: exp_col = cols[k]; break

qtext_col = None
for k in ["question_text","question","prompt","stem","item_stem","problem","task"]:
    if k in cols: qtext_col = cols[k]; break

print("Detected:")
print(" category:", cat_col)
print(" misconception:", mis_col)
print(" qid:", qid_col)
print(" answer:", ans_col)
print(" explanation:", exp_col)
print(" question text:", qtext_col)

train_df["label_str"] = train_df[cat_col].astype(str) + ":" + train_df[mis_col].astype(str)
train_df.head(3)


train: /kaggle/input/datasets/mkhanshovon43/map-ds/DS/train.csv (36696, 7)
cols: ['row_id', 'QuestionId', 'QuestionText', 'MC_Answer', 'StudentExplanation', 'Category', 'Misconception']
Detected:
 category: Category
 misconception: Misconception
 qid: QuestionId
 answer: MC_Answer
 explanation: None
 question text: None


,row_id,QuestionId,QuestionText,MC_Answer,StudentExplanation,Category,Misconception,label_str
0,0,31772,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),0ne third is equal to tree nineth,True_Correct,NaN,True_Correct:nan
1,1,31772,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),1 / 3 because 6 over 9 is 2 thirds and 1 third...,True_Correct,NaN,True_Correct:nan
2,2,31772,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),"1 3rd is half of 3 6th, so it is simplee to un...",True_Neither,NaN,True_Neither:nan


Cell 3 — Infer correct answer per question + build model input text

In [4]:
import numpy as np

def build_correct_by_q(df: pd.DataFrame) -> dict:
    if qid_col is None or ans_col is None:
        return {}
    tcat = df[cat_col].astype(str)
    correct_mask = tcat.str.startswith("True", na=False) | (tcat == "True_Correct")
    tmp = df.loc[correct_mask, [qid_col, ans_col]].dropna()
    if len(tmp) == 0:
        return {}
    return tmp.groupby(qid_col)[ans_col].agg(lambda s: s.value_counts().index[0]).to_dict()

correct_by_q_full = build_correct_by_q(train_df)
print("Correct answers inferred:", len(correct_by_q_full))

def compute_is_correct(df: pd.DataFrame, correct_by_q: dict) -> np.ndarray:
    if qid_col is None or ans_col is None or not correct_by_q:
        return np.full(len(df), -1, dtype=np.int8)
    out = np.full(len(df), -1, dtype=np.int8)
    for i, (q, a) in enumerate(zip(df[qid_col].values, df[ans_col].values)):
        if pd.isna(q) or pd.isna(a): 
            continue
        if q in correct_by_q:
            out[i] = 1 if str(a) == str(correct_by_q[q]) else 0
    return out

train_df["is_correct"] = compute_is_correct(train_df, correct_by_q_full)
print(train_df["is_correct"].value_counts(dropna=False))

def safe_col(df, col):
    if col is None or col not in df.columns:
        return pd.Series([""] * len(df))
    return df[col].fillna("").astype(str)

def make_text(df: pd.DataFrame, is_correct: np.ndarray) -> list:
    q = safe_col(df, qtext_col)
    a = safe_col(df, ans_col)
    e = safe_col(df, exp_col)
    ic = pd.Series(is_correct).map({1:"Yes", 0:"No"}).fillna("Unknown").astype(str)
    return ("Question: " + q + "\nAnswer: " + a + "\nIs Correct Answer: " + ic + "\nStudent Explanation: " + e).tolist()

train_text_full = make_text(train_df, train_df["is_correct"].values.astype(np.int8))
print(train_text_full[0][:500])


Correct answers inferred: 15
is_correct
1    20464
0    16232
Name: count, dtype: int64
Question: 
Answer: \( \frac{1}{3} \)
Is Correct Answer: Yes
Student Explanation: 


Cell 4 — Train DeBERTa + CV MAP@3 (mean score on train)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import GroupKFold
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding, set_seed,
    get_linear_schedule_with_warmup, get_cosine_schedule_with_warmup
)
from datasets import Dataset
from torch.optim import AdamW
import copy

classes = np.array(sorted(train_df["label_str"].unique().tolist()))
lab2id = {c:i for i,c in enumerate(classes)}
id2lab = {i:c for c,i in lab2id.items()}
y_all = train_df["label_str"].map(lab2id).astype(int).values
num_labels = len(classes)
print("num_labels:", num_labels)

true_mask  = np.array([c.startswith("True")  for c in classes], dtype=bool)
false_mask = np.array([c.startswith("False") for c in classes], dtype=bool)

# ============================================================
# ADVANCED TECHNIQUES FOR MAX ACCURACY
# ============================================================

class AWP:
    """Adversarial Weight Perturbation - significantly boosts accuracy"""
    def __init__(self, model, adv_lr=0.1, adv_eps=1e-4):
        self.model = model
        self.adv_lr = adv_lr
        self.adv_eps = adv_eps
        self.backup = {}
        self.backup_eps = {}

    def attack_step(self):
        e = 1e-6
        for name, param in self.model.named_parameters():
            if param.requires_grad and param.grad is not None:
                norm1 = torch.norm(param.grad)
                norm2 = torch.norm(param.data.detach())
                if norm1 != 0 and not torch.isnan(norm1):
                    r_at = self.adv_lr * param.grad / (norm1 + e) * (norm2 + e)
                    param.data.add_(r_at)
                    param.data = torch.min(
                        torch.max(param.data, self.backup_eps[name][0]), 
                        self.backup_eps[name][1]
                    )

    def save(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and param.grad is not None:
                self.backup[name] = param.data.clone()
                grad_eps = self.adv_eps * param.abs().detach()
                self.backup_eps[name] = (
                    self.backup[name] - grad_eps,
                    self.backup[name] + grad_eps,
                )

    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}
        self.backup_eps = {}

class EMA:
    """Exponential Moving Average - smooths model weights"""
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        self._register()

    def _register(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()

    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data
                param.data = self.shadow[name]

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]
        self.backup = {}

def compute_kl_loss(p, q, pad_mask=None):
    """KL divergence for R-Drop"""
    p_loss = F.kl_div(F.log_softmax(p, dim=-1), F.softmax(q, dim=-1), reduction='none')
    q_loss = F.kl_div(F.log_softmax(q, dim=-1), F.softmax(p, dim=-1), reduction='none')
    if pad_mask is not None:
        p_loss = p_loss.masked_fill(pad_mask.unsqueeze(-1), 0.)
        q_loss = q_loss.masked_fill(pad_mask.unsqueeze(-1), 0.)
    p_loss = p_loss.sum()
    q_loss = q_loss.sum()
    return (p_loss + q_loss) / 2

class LabelSmoothingCrossEntropy(nn.Module):
    """Label smoothing for better generalization"""
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, pred, target):
        n_classes = pred.size(-1)
        with torch.no_grad():
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (n_classes - 1))
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
        return torch.mean(torch.sum(-true_dist * F.log_softmax(pred, dim=-1), dim=-1))

def get_optimizer_grouped_parameters(model, learning_rate, weight_decay, layerwise_lr_decay):
    """Layerwise learning rate decay - lower LR for earlier layers"""
    no_decay = ["bias", "LayerNorm.weight", "LayerNorm.bias"]
    
    # Get the number of layers
    num_layers = model.config.num_hidden_layers
    layers = [model.deberta.embeddings] + list(model.deberta.encoder.layer)
    layers.reverse()
    
    optimizer_grouped_parameters = []
    for i, layer in enumerate(layers):
        lr = learning_rate * (layerwise_lr_decay ** i)
        optimizer_grouped_parameters += [
            {
                "params": [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)],
                "weight_decay": weight_decay,
                "lr": lr,
            },
            {
                "params": [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)],
                "weight_decay": 0.0,
                "lr": lr,
            },
        ]
    
    # Classifier head uses full LR
    optimizer_grouped_parameters += [
        {
            "params": [p for n, p in model.classifier.named_parameters() if not any(nd in n for nd in no_decay)],
            "weight_decay": weight_decay,
            "lr": learning_rate,
        },
        {
            "params": [p for n, p in model.classifier.named_parameters() if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
            "lr": learning_rate,
        },
    ]
    
    return optimizer_grouped_parameters

def apply_filters(logits: np.ndarray, is_correct: np.ndarray, allowed_mask=None) -> np.ndarray:
    out = logits.copy()
    # Apply temperature scaling
    if USE_TEMPERATURE_SCALING:
        out = out / TEMPERATURE
    big_neg = -1e9
    if USE_FAMILY_FILTER:
        for i, ic in enumerate(is_correct):
            if ic == 1: out[i, ~true_mask] = big_neg
            elif ic == 0: out[i, ~false_mask] = big_neg
    if allowed_mask is not None:
        out[~allowed_mask] = big_neg
    return out

def top3_from_logits(logits: np.ndarray):
    idx = np.argsort(-logits, axis=1)[:, :10]
    out = []
    for row in idx:
        seen, picks = set(), []
        for j in row:
            if j in seen: 
                continue
            seen.add(int(j))
            picks.append(int(j))
            if len(picks) == 3:
                break
        while len(picks) < 3:
            picks.append(int(row[len(picks)]))
        out.append(picks)
    return out

def map_at_3(y_true: np.ndarray, y_pred_top3):
    score = 0.0
    for yt, preds in zip(y_true, y_pred_top3):
        used = set()
        for rank, p in enumerate(preds[:3], start=1):
            if p in used: 
                continue
            used.add(p)
            if p == yt:
                score += 1.0 / rank
                break
    return score / len(y_true)

tokenizer = AutoTokenizer.from_pretrained(CLASSIFIER_MODEL, use_fast=True)
collator = DataCollatorWithPadding(tokenizer)

groups = train_df[qid_col].values if qid_col is not None else np.arange(len(train_df))
gkf = GroupKFold(n_splits=N_FOLDS)

# Store all fold logits for ensemble
all_oof_logits = np.zeros((len(train_df), num_labels, len(SEEDS)), dtype=np.float32)
fold_scores = []
oof_top3_labels = [None] * len(train_df)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

for seed_idx, seed in enumerate(SEEDS):
    set_seed(seed)
    print(f"\n{'='*60}")
    print(f" SEED {seed} ({seed_idx+1}/{len(SEEDS)})")
    print(f"{'='*60}")

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(np.zeros(len(train_df)), y_all, groups), start=1):
        tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
        va_df = train_df.iloc[va_idx].reset_index(drop=True)

        # avoid leakage: infer correct answers from TRAIN fold only
        correct_by_q = build_correct_by_q(tr_df)
        tr_is_correct = compute_is_correct(tr_df, correct_by_q)
        va_is_correct = compute_is_correct(va_df, correct_by_q)

        allowed_mask_va = None
        if USE_QID_LABEL_RESTRICTION and qid_col is not None:
            allowed_by_q = {}
            tr_y = tr_df["label_str"].map(lab2id).values
            for q, lab in zip(tr_df[qid_col].values, tr_y):
                allowed_by_q.setdefault(q, set()).add(int(lab))
            allowed_mask_va = np.zeros((len(va_df), num_labels), dtype=bool)
            for i, q in enumerate(va_df[qid_col].values):
                if q in allowed_by_q:
                    allowed_mask_va[i, list(allowed_by_q[q])] = True
                else:
                    allowed_mask_va[i, :] = True

        tr_text = make_text(tr_df, tr_is_correct)
        va_text = make_text(va_df, va_is_correct)

        tr_y = tr_df["label_str"].map(lab2id).astype(int).values
        va_y = va_df["label_str"].map(lab2id).astype(int).values

        # Create datasets WITHOUT is_correct (keep it as separate numpy arrays for filtering)
        tr_ds = Dataset.from_dict({"text": tr_text, "labels": tr_y})
        va_ds = Dataset.from_dict({"text": va_text, "labels": va_y})

        def tok(batch): return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)
        tr_ds = tr_ds.map(tok, batched=True, remove_columns=["text"]).with_format("torch")
        va_ds = va_ds.map(tok, batched=True, remove_columns=["text"]).with_format("torch")

        # Load model
        model = AutoModelForSequenceClassification.from_pretrained(CLASSIFIER_MODEL, num_labels=num_labels)
        model.to(device)
        
        # Setup optimizer with layerwise LR decay
        if USE_LAYERWISE_LR_DECAY:
            optimizer_grouped_params = get_optimizer_grouped_parameters(
                model, LR, WEIGHT_DECAY, LAYERWISE_LR_DECAY
            )
            optimizer = AdamW(optimizer_grouped_params, lr=LR, eps=1e-8)
        else:
            optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, eps=1e-8)
        
        # Setup scheduler
        num_training_steps = int(len(tr_ds) / (BATCH_SIZE * GRAD_ACCUM) * EPOCHS)
        num_warmup_steps = int(num_training_steps * WARMUP_RATIO)
        scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)
        
        # Setup loss function
        if LABEL_SMOOTHING > 0:
            criterion = LabelSmoothingCrossEntropy(smoothing=LABEL_SMOOTHING)
        else:
            criterion = nn.CrossEntropyLoss()
        
        # Setup AWP and EMA
        awp = AWP(model, adv_lr=AWP_LR, adv_eps=AWP_EPS) if USE_AWP else None
        ema = EMA(model, decay=EMA_DECAY) if USE_EMA else None
        
        # Create dataloader
        train_loader = torch.utils.data.DataLoader(
            tr_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator
        )
        
        print(f"\n--- Fold {fold}/{N_FOLDS} | Steps: {num_training_steps} | Warmup: {num_warmup_steps} ---")
        
        # Training loop
        model.train()
        global_step = 0
        optimizer.zero_grad()
        
        for epoch in range(EPOCHS):
            epoch_loss = 0
            for step, batch in enumerate(train_loader):
                batch = {k: v.to(device) for k, v in batch.items()}
                # Extract labels and remove keys not accepted by model
                labels = batch.pop("labels")
                batch.pop("is_correct", None)
                batch.pop("label", None)
                
                # Forward pass
                outputs = model(**batch)
                logits = outputs.logits
                
                # R-Drop: forward twice and add KL loss
                if USE_RDROP:
                    outputs2 = model(**batch)
                    logits2 = outputs2.logits
                    ce_loss = (criterion(logits, labels) + criterion(logits2, labels)) / 2
                    kl_loss = compute_kl_loss(logits, logits2)
                    loss = ce_loss + RDROP_ALPHA * kl_loss
                else:
                    loss = criterion(logits, labels)
                
                loss = loss / GRAD_ACCUM
                loss.backward()
                epoch_loss += loss.item() * GRAD_ACCUM
                
                if (step + 1) % GRAD_ACCUM == 0:
                    # Gradient clipping
                    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                    
                    # AWP attack step
                    if USE_AWP and epoch >= AWP_START_EPOCH:
                        awp.save()
                        awp.attack_step()
                        
                        # Forward pass with perturbed weights
                        outputs_adv = model(**batch)
                        loss_adv = criterion(outputs_adv.logits, labels)
                        loss_adv = loss_adv / GRAD_ACCUM
                        loss_adv.backward()
                        
                        awp.restore()
                    
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
                    
                    # EMA update
                    if USE_EMA:
                        ema.update()
                    
                    global_step += 1
            
            avg_loss = epoch_loss / len(train_loader)
            print(f"  Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")
        
        # Apply EMA weights for evaluation
        if USE_EMA:
            ema.apply_shadow()
        
        # Evaluation
        model.eval()
        val_loader = torch.utils.data.DataLoader(
            va_ds, batch_size=BATCH_SIZE*2, shuffle=False, collate_fn=collator
        )
        
        all_logits = []
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                # Remove keys not accepted by model
                batch.pop("labels", None)
                batch.pop("label", None)
                batch.pop("is_correct", None)
                outputs = model(**batch)
                all_logits.append(outputs.logits.cpu().numpy())
        
        va_logits = np.concatenate(all_logits, axis=0)
        
        # Restore original weights after EMA eval
        if USE_EMA:
            ema.restore()
        
        # Store logits for ensemble
        for ii, ridx in enumerate(va_idx):
            all_oof_logits[ridx, :, seed_idx] = va_logits[ii]
        
        va_logits_filtered = apply_filters(va_logits, va_is_correct, allowed_mask_va)
        va_top3 = top3_from_logits(va_logits_filtered)
        fold_map3 = map_at_3(va_y, va_top3)
        fold_scores.append(fold_map3)
        print(f"  ✓ Fold MAP@3: {fold_map3:.6f}")

        # Store predictions for this fold
        for ii, ridx in enumerate(va_idx):
            oof_top3_labels[ridx] = [id2lab[j] for j in va_top3[ii]]
        
        # Clean up
        del model, optimizer, scheduler, awp, ema
        torch.cuda.empty_cache()
        import gc; gc.collect()

# Final ensemble: average logits across seeds
print(f"\n{'='*60}")
print(" FINAL RESULTS (MULTI-SEED ENSEMBLE)")
print(f"{'='*60}")

ensemble_logits = all_oof_logits.mean(axis=2)

# Recompute predictions with ensembled logits
is_correct_full = train_df["is_correct"].values
ensemble_logits_filtered = apply_filters(ensemble_logits, is_correct_full)
ensemble_top3 = top3_from_logits(ensemble_logits_filtered)
ensemble_map3 = map_at_3(y_all, ensemble_top3)

# Update predictions with ensemble
for i in range(len(train_df)):
    oof_top3_labels[i] = [id2lab[j] for j in ensemble_top3[i]]

print(f"Individual fold scores: {[round(s, 5) for s in fold_scores]}")
print(f"Mean fold MAP@3: {np.mean(fold_scores):.6f}")
print(f"✅ ENSEMBLE MAP@3: {ensemble_map3:.6f}")
train_df[["label_str","pred_top3"]].head(5)

train_df["pred_top3"] = [" ".join(p) for p in oof_top3_labels]

train_df[["label_str","pred_top3"]].head(5)

2026-02-19 13:05:46.871975: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771506347.192754     279 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771506347.277778     279 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771506348.123790     279 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771506348.123820     279 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771506348.123822     279 computation_placer.cc:177] computation placer alr

num_labels: 65


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:558: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


... (29 outputs left out)


--- Fold 5/5 ---


Step,Training Loss
100,2.217500
200,1.366200
300,1.278500
400,1.090600
500,0.995900
600,0.977300
700,0.943800
800,0.917900
900,0.887200
1000,0.857800


Fold MAP@3: 0.204276

==================== TRAIN RESULTS ====================
Fold scores: [0.075093, 0.139542, 0.167757, 0.148616, 0.204276]
Mean MAP@3 : 0.147057


,label_str,pred_top3
0,True_Correct:nan,False_Neither:nan False_Misconception:SwapDivi...
1,True_Correct:nan,False_Neither:nan False_Misconception:SwapDivi...
2,True_Neither:nan,False_Neither:nan False_Misconception:SwapDivi...
3,True_Neither:nan,False_Neither:nan False_Misconception:SwapDivi...
4,True_Correct:nan,False_Neither:nan False_Misconception:SwapDivi...


Cell 5 — Load Explainer LLM (Qwen 7B 4-bit) + helper generate()

In [6]:
import torch

def load_explainer():
    if not USE_LLM_EXPLAINER:
        return None, None, None

    from transformers import (
        AutoTokenizer,
        AutoModelForCausalLM,
        AutoModelForSeq2SeqLM,
        BitsAndBytesConfig
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Best: 4-bit causal LLM (GPU)
    if device == "cuda":
        try:
            bnb_cfg = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
            )
            tok = AutoTokenizer.from_pretrained(EXPLAINER_MODEL, use_fast=True)
            mdl = AutoModelForCausalLM.from_pretrained(
                EXPLAINER_MODEL,
                quantization_config=bnb_cfg,
                device_map="auto",
                torch_dtype=torch.float16,
            )
            mdl.eval()
            print("✅ Loaded explainer (4-bit):", EXPLAINER_MODEL)
            return "causal", mdl, tok
        except Exception as e:
            print("⚠️ 4-bit causal explainer failed:", repr(e)[:200])

    # Fallback: FLAN-T5 (CPU/GPU)
    tok = AutoTokenizer.from_pretrained(EXPLAINER_FALLBACK_MODEL, use_fast=True)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(EXPLAINER_FALLBACK_MODEL).to(device)
    mdl.eval()
    print("✅ Loaded explainer fallback:", EXPLAINER_FALLBACK_MODEL, "on", device)
    return "seq2seq", mdl, tok

EXPLAINER_KIND, explainer_model, explainer_tok = load_explainer()

def llm_generate(prompt: str, max_new_tokens=320):
    if explainer_model is None:
        return ""
    if EXPLAINER_KIND == "causal":
        inputs = explainer_tok(prompt, return_tensors="pt", truncation=True, max_length=2048).to(explainer_model.device)
        with torch.no_grad():
            out = explainer_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=0.0,
                num_beams=1,
            )
        text = explainer_tok.decode(out[0], skip_special_tokens=True)
        return text[len(prompt):].strip() if text.startswith(prompt) else text.strip()
    else:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        inputs = explainer_tok(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
        with torch.no_grad():
            out = explainer_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, num_beams=4)
        return explainer_tok.decode(out[0], skip_special_tokens=True).strip()


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

... (3 outputs left out)

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Loaded explainer (4-bit): Qwen/Qwen2.5-7B-Instruct


Cell 6 — Build Misconception Knowledge Base (KB) using the LLM (cached)

In [7]:
import json, re
from collections import Counter

train_df["miscon_only"] = train_df[mis_col].astype(str)
freq = Counter(train_df["miscon_only"].tolist())
top_miscons = [m for m,_ in freq.most_common(KB_BUILD_TOP_N)]
print("KB target:", len(top_miscons), "most common:", freq.most_common(5))

if KB_PATH.exists():
    kb = json.loads(KB_PATH.read_text(encoding="utf-8"))
    print("✅ Loaded KB entries:", len(kb))
else:
    kb = {}
    for i, miscon in enumerate(top_miscons, start=1):
        sub = train_df[train_df["miscon_only"] == miscon]
        sample = sub.sample(n=min(KB_EXAMPLES_PER_MISCON, len(sub)), random_state=42)

        examples = []
        for _, r in sample.iterrows():
            examples.append({
                "question": str(r[qtext_col]) if qtext_col else "",
                "student_answer": str(r[ans_col]) if ans_col else "",
                "student_expl": str(r[exp_col]) if exp_col else "",
            })

        prompt = f"""You are a math teacher writing a misconception knowledge card.

Misconception name: {miscon}

Student examples (question + reasoning):
{json.dumps(examples, ensure_ascii=False, indent=2)}

Return ONLY JSON with keys:
plain_definition (1-2 sentences),
typical_wrong_thought (list),
correct_rule (list),
fix_steps (list of 4-7 steps),
quick_check (one question),
tiny_example (2-4 lines).
"""
        out = llm_generate(prompt, max_new_tokens=420) if explainer_model is not None else ""
        card = None
        if out:
            m = re.search(r"\{.*\}", out, flags=re.S)
            if m:
                try:
                    card = json.loads(m.group(0))
                except Exception:
                    card = None

        if card is None:
            card = {
                "plain_definition": f"Students often make this mistake: {miscon}.",
                "typical_wrong_thought": ["They apply an incorrect rule or shortcut."],
                "correct_rule": ["Use the correct definition and check carefully."],
                "fix_steps": ["Restate the goal.", "Write the correct rule.", "Apply it step-by-step.", "Check with an easy example."],
                "quick_check": "Can you test your rule on a simpler number?",
                "tiny_example": "Example: Compare decimals by place value, not by number of digits."
            }

        kb[miscon] = card
        if i % 10 == 0:
            print(f"Built {i}/{len(top_miscons)} cards...")

    KB_PATH.write_text(json.dumps(kb, ensure_ascii=False, indent=2), encoding="utf-8")
    print("✅ Saved KB:", len(kb), "to", KB_PATH)

k0 = next(iter(kb.keys()))
print("Sample KB:", k0)
print(json.dumps(kb[k0], ensure_ascii=False, indent=2)[:1200])


KB target: 36 most common: [('nan', 26836), ('Incomplete', 1454), ('Additive', 929), ('Duplication', 704), ('Subtraction', 620)]


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:623: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
Starting from v4.46, the `logits` model output will have the same type as the model (except at train 

Built 10/36 cards...
Built 20/36 cards...
Built 30/36 cards...
✅ Saved KB: 36 to misconception_kb.json
Sample KB: nan
{
  "plain_definition": "NaN is not a number and represents an undefined or unrepresentable value in mathematics. Students often confuse it with other numerical values or fractions.",
  "typical_wrong_thought": [
    "NaN is a numerical value like any other number.",
    "NaN can be represented as a fraction.",
    "NaN is the same as infinity.",
    "NaN can be a whole number."
  ],
  "correct_rule": [
    "NaN is not a valid number and cannot be used in arithmetic operations.",
    "NaN should not be confused with infinity, which is a concept representing unboundedness.",
    "NaN does not represent a fraction or a specific numerical value."
  ],
  "fix_steps": [
    "Understand that NaN stands for 'Not a Number' and is used to indicate an undefined or unrepresentable value.",
    "Recognize that NaN cannot be used in arithmetic operations and will result in NaN if us

Cell 7 — Explainable AI Feedback Generator + Verifier

In [ ]:
import numpy as np
import re
import json

def parse_top3(s: str):
    parts = str(s).split()
    return parts[:3] if parts else []

def get_correct_choice(qid):
    if qid_col is None or ans_col is None:
        return ""
    return str(correct_by_q_full.get(qid, ""))

def rule_based_verify(feedback: str, is_correct: int, correct_choice: str) -> str:
    txt = feedback.strip()

    # If the student is correct, don't say "wrong/incorrect"
    if is_correct == 1:
        txt = re.sub(r"\bincorrect\b", "not quite right", txt, flags=re.I)
        txt = re.sub(r"\bwrong\b", "not quite right", txt, flags=re.I)

    # If student is incorrect and correct choice is known, ensure we mention it
    if is_correct == 0 and correct_choice and correct_choice not in txt:
        txt = f"✅ Correct answer choice: {correct_choice}\n\n" + txt

    return txt

def llm_verify_and_fix(feedback: str) -> str:
    if (not USE_LLM_VERIFIER) or (explainer_model is None):
        return feedback

    prompt = f"""You are a strict teacher-editor.

Fix contradictions and make the feedback concise (8–14 lines), student-friendly, and correct.

Feedback:
{feedback}
"""
    out = llm_generate(prompt, max_new_tokens=260)
    return out if out else feedback

def generate_teacher_feedback(row, pred_label: str, pred_top3: list) -> str:
    qtext = str(row[qtext_col]) if qtext_col else ""
    student_answer = str(row[ans_col]) if ans_col else ""
    student_expl = str(row[exp_col]) if exp_col else ""
    qid = row[qid_col] if qid_col else None

    correct_choice = get_correct_choice(qid) if qid is not None else ""
    ic = int(row["is_correct"]) if "is_correct" in row else -1
    ic_str = {1:"Yes", 0:"No"}.get(ic, "Unknown")

    miscon_only = pred_label.split(":", 1)[1] if ":" in pred_label else pred_label
    card = kb.get(miscon_only, {})
    card_json = json.dumps(card, ensure_ascii=False, indent=2)

    if USE_LLM_EXPLAINER and explainer_model is not None:
        prompt = f"""You are a kind, clear math teacher.

Use ONLY the info below. Do not invent details.

Question: {qtext}
Student chose: {student_answer}
Is correct? {ic_str}
Correct choice (if known): {correct_choice}
Student explanation: {student_expl}

Top predictions: {pred_top3}

Misconception knowledge card (JSON):
{card_json}

Write:
- 1–2 lines: what the student seems to think
- 1 line: misconception in simple words
- 4–7 short steps: correct logic
- 1 line: self-check question

Keep it concise and encouraging.
"""
        fb = llm_generate(prompt, max_new_tokens=320) or "LLM failed to generate feedback."
    else:
        fb = f"""Teacher Feedback (template)
You chose: {student_answer}. Correct? {ic_str}. Correct choice: {correct_choice}
Likely misconception: {pred_label}

Steps:
1) Restate what is asked.
2) Write the correct rule.
3) Apply it step-by-step.
4) Check using a simple example.

Self-check: Does your rule still work if numbers change slightly?
"""

    fb = rule_based_verify(fb, ic, correct_choice)
    fb = llm_verify_and_fix(fb)
    return fb.strip()

# Demo: show explanations for 5 random rows
idxs = np.random.default_rng(42).choice(len(train_df), size=5, replace=False)
for i in idxs:
    row = train_df.iloc[i]
    top3 = parse_top3(row["pred_top3"])
    pred1 = top3[0] if top3 else "Unknown:Unknown"
    
    print("="*100)
    print(f"📋 ROW {i} DATA")
    print("="*100)
    
    # Print row data in formatted way
    if qid_col:
        print(f"Question ID: {row[qid_col]}")
    if qtext_col:
        print(f"Question Text: {row[qtext_col]}")
    if ans_col:
        print(f"Student Answer: {row[ans_col]}")
    if exp_col:
        print(f"Student Explanation: {row[exp_col]}")
    if cat_col:
        print(f"Category: {row[cat_col]}")
    if mis_col:
        print(f"Misconception: {row[mis_col]}")
    
    correct_choice = get_correct_choice(row[qid_col]) if qid_col else ""
    if correct_choice:
        print(f"Correct Answer: {correct_choice}")
    print(f"Is Correct: {row.get('is_correct', 'Unknown')}")
    print(f"True Label: {row['label_str']}")
    
    print("\n" + "-"*100)
    print("🤖 DEBERTA PREDICTIONS (Top 3)")
    print("-"*100)
    for rank, pred in enumerate(top3, start=1):
        marker = "✓" if pred == row['label_str'] else " "
        print(f"{marker} {rank}. {pred}")
    
    print("\n" + "-"*100)
    print("📚 TEACHER FEEDBACK & EXPLANATION")
    print("-"*100)
    print(generate_teacher_feedback(row, pred1, top3))
    print("\n")


Row: 28398
True: False_Misconception:Unknowable
Pred: ['False_Neither:nan', 'False_Misconception:Scale', 'False_Misconception:Whole_numbers_larger']

- Next steps: Review the concept of NaN and its usage in mathematical and programming contexts. Practice identifying and handling NaN values in various scenarios. 

---

Feedback:
✅ Correct answer choice: \( 10 \)

- What the student seems to think: The student believes there isn't enough information provided to answer the question.
- Misconception in simple words: Thinking NaN is a number or fraction.
- Correct logic:
  1. Understand that NaN stands for 'Not a Number' and indicates an undefined value.
  2. Recognize that NaN cannot be used in arithmetic operations.
  3. Avoid confusing NaN with other numerical values or fractions.
  4. Use NaN only when it is appropriate to indicate an undefined or unrepresentable value.
  5. Check your calculations to ensure you are not mistakenly using NaN where a numerical value is expected.
- Self-ch